# Showjumping SSL — Milestone Notebook

End-to-end pipeline runnable on Colab Pro. Clones the code repo from GitHub, mounts Drive for the (large) data and checkpoint artifacts, runs scrape → segment → YOLO filter → annotate → geometric d → SSL pretraining → embedding → figures.

**Runtime:** A100 or L4 is fastest (SSL training fits comfortably at batch 16). T4 (16 GB) works but you'll want to drop batch to 4 in cell 5.

**One-time edits before running:** set `GITHUB_REPO` and `DRIVE_DATA_ROOT` in cell 0 to your own values.

## 0. Clone repo + mount Drive

Layout after this cell:
```
/content/project/                       <- cloned from GitHub (fresh each session)
├── src/, notebooks/, milestone_overleaf/, requirements.txt
├── data/         -> symlink to DRIVE_DATA_ROOT/data
└── checkpoints/  -> symlink to DRIVE_DATA_ROOT/checkpoints
```

In [ ]:
import os, sys, subprocess
from pathlib import Path

GITHUB_REPO = 'https://github.com/bballhaus/showjumping-ssl.git'
DRIVE_DATA_ROOT = '/content/drive/MyDrive/CS131'
BRANCH = 'main'

from google.colab import drive
drive.mount('/content/drive')
Path(f'{DRIVE_DATA_ROOT}/data').mkdir(parents=True, exist_ok=True)
Path(f'{DRIVE_DATA_ROOT}/checkpoints').mkdir(parents=True, exist_ok=True)

REPO_DIR = Path('/content/project')
if REPO_DIR.exists():
    subprocess.run(['git', '-C', str(REPO_DIR), 'fetch', '--all'], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'reset', '--hard', f'origin/{BRANCH}'], check=True)
else:
    subprocess.run(['git', 'clone', '--branch', BRANCH, '--depth', '1',
                    GITHUB_REPO, str(REPO_DIR)], check=True)

for sub in ['data', 'checkpoints']:
    target = Path(f'{DRIVE_DATA_ROOT}/{sub}')
    link = REPO_DIR / sub
    if link.is_symlink() or link.exists():
        if link.is_dir() and not link.is_symlink():
            subprocess.run(['rm', '-rf', str(link)], check=True)
        else:
            link.unlink(missing_ok=True)
    link.symlink_to(target, target_is_directory=True)

os.chdir(REPO_DIR)
sys.path.insert(0, str(REPO_DIR))
print('cwd:', os.getcwd())
print('data ->', os.readlink('data'))
print('checkpoints ->', os.readlink('checkpoints'))

## 1. Install dependencies + verify GPU

`requirements.txt` deliberately does *not* pin torch / torchvision — Colab's base image ships them with CUDA support, and installing from PyPI would silently replace them with the CPU wheel.

In [ ]:
!pip install -q -r requirements.txt
!apt-get -qq install -y ffmpeg

import torch
print(f'torch: {torch.__version__} | cuda: {torch.cuda.is_available()}')
!nvidia-smi 2>&1 | head -10

## 2. Scrape + segment + filter clips

Curate `src/data/clip_sources.txt` with YouTube URLs (FEI World Cup, Longines GCT, CSI 3* highlight reels at 1.50 m) before running.

- yt-dlp prints its own progress bar for downloads (~5–10 min per 2-hour broadcast).
- `segment_video` shows a tqdm bar per video, then a nested per-clip bar.
- `filter_clips` runs YOLO on each clip and moves non-jumping ones (crowd shots, course walks, interviews) into `data/clips_rejected/`.

In [ ]:
import time
from pathlib import Path
from tqdm.auto import tqdm

from src.data.scrape import read_sources, download_one, DEFAULT_SOURCES
from src.data.segment import segment_video
from src.data.filter_clips import filter_clips

RAW = Path('data/raw')
CLIPS = Path('data/clips')
REJECTED = Path('data/clips_rejected')
CLIP_LEN = 2.0
STRIDE = 30.0
MIN_HORSE_FRAC = 0.4

# 2a. Download.
urls = read_sources(DEFAULT_SOURCES)
print(f'[scrape] {len(urls)} videos to download -> {RAW}')
for url in tqdm(urls, desc='downloads', unit='vid'):
    download_one(url, RAW)

# 2b. Segment.
vids = sorted(RAW.glob('*.mp4'))
print(f'[segment] cutting {len(vids)} videos at stride={STRIDE}s ...')
total = 0
t0 = time.time()
for v in tqdm(vids, desc='segment', unit='vid'):
    n = segment_video(v, CLIPS, clip_len=CLIP_LEN, stride=STRIDE)
    tqdm.write(f'  {v.name}: {n} clips')
    total += n
print(f'[segment] {total} raw clips in {time.time()-t0:.0f}s')

# 2c. YOLO horse-presence filter.
raw_count = len(list(CLIPS.glob('*.mp4')))
print(f'[filter] running YOLO on {raw_count} clips ...')
kept, rejected = filter_clips(CLIPS, REJECTED,
                              weights='yolov8n.pt',
                              device='cuda',
                              stride=4,
                              min_frac=MIN_HORSE_FRAC,
                              conf=0.35)
print(f'[filter] kept {kept}, rejected {rejected}')

## 3. Hand-annotate fence boxes

Inline matplotlib + ipywidgets annotator. Use the frame slider to scrub to a frame where the fence is clearly visible (broadcast cameras track the horse, not the fence), drag a rectangle around the fence, then click **Vertical** (1 pole) or **Oxer** (2 poles). Skip clips that aren't clean side-on views.

~25–40 boxes covering all source videos is the milestone target. Annotations write to `data/annotations/fences.csv` on Drive in real time and resume across sessions.

In [ ]:
!pip install -q ipympl ipywidgets
from google.colab import output
output.enable_custom_widget_manager()
%matplotlib widget

import random
from src.preprocess.annotate_colab import ColabAnnotator

ann = ColabAnnotator(
    clips_dir='data/clips',
    out_csv='data/annotations/fences.csv',
    limit=200,   # candidate pool; we shuffle and take 100 to avoid re-seeing skipped clips
)
random.Random(42).shuffle(ann.queue)
ann.queue = ann.queue[:100]
ann.start()

## 4. YOLO horse detection + geometric d on all clips

Walks every clip: runs YOLO on every 2nd frame, finds the takeoff frame (vertical-velocity reversal of the horse-box bottom edge), and computes `d_meters` for clips that have a hand-drawn fence box. Clips without a fence annotation get `d_meters = NaN`.

In [ ]:
from pathlib import Path
import pandas as pd
from tqdm.auto import tqdm
from src.preprocess.run_pipeline import process_clip, load_fence_annotations
from src.preprocess.detect import HorseDetector

detector = HorseDetector(weights='yolov8n.pt', device='cuda')
fences = load_fence_annotations(Path('data/annotations/fences.csv'))
clips = sorted(Path('data/clips').glob('*.mp4'))
print(f'[pipeline] {len(clips)} clips, {len(fences)} fence annotations')

rows = []
for cp in tqdm(clips, desc='YOLO + geometry', unit='clip'):
    rows.append(process_clip(cp, detector, fences.get(cp.stem)))

df = pd.DataFrame(rows)
out = Path('data/annotations/auto.csv')
out.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(out, index=False)
print(f'wrote {len(df)} rows -> {out}')
print(df[df['d_meters'].notna()][['clip_id', 'type', 'd_meters']].head(10))

## 5. SSL pretraining

R(2+1)D-18 trunk + 2-layer MLP projector. Joint InfoNCE (on two augmented views) + 6-way temporal-order classification (on 3 disjoint sub-clips). Cosine-decayed AdamW.

- **A100 / L4:** batch 16, ~10 min for 10 epochs.
- **T4 (16 GB):** drop `batch_size` to 4 *or* use the AMP inline version below.

If retrieval in cell 7 shows collapse (similarities ≈ 0.97 across unrelated clips), skip the rest of this cell and use the Kinetics-fallback cell 7b.

In [ ]:
from pathlib import Path
from src.ssl.train import train

train(
    clips_dir=Path('data/clips'),
    out_dir=Path('checkpoints'),
    epochs=10,
    batch_size=16,           # drop to 4 on T4
    lr=1e-3,
    tau=0.1,
    lambda_order=0.3,
    n_workers=2,
    device='cuda',
    kinetics_init=False,     # set True to skip from-scratch and go straight to fine-tuning Kinetics
)

## 6. Generate embeddings

In [ ]:
from pathlib import Path
from src.ssl.embed import embed_dir

embed_dir(
    clips_dir=Path('data/clips'),
    ckpt_path=Path('checkpoints/encoder.pt'),
    out_npz=Path('data/embeddings.npz'),
    device='cuda',
    batch_size=16,
    kinetics_init=False,
)

## 7. Nearest-neighbor sanity check

If the 5 nearest neighbors of a query clip share fence type, approach phase, or camera angle — the encoder learned something useful. If similarities are uniformly > 0.95 across unrelated clips, the encoder has collapsed and you should run cell 7b (Kinetics fallback).

In [ ]:
import numpy as np, random
data = np.load('data/embeddings.npz', allow_pickle=True)
F, paths = data['features'], list(data['paths'])
F_n = F / (np.linalg.norm(F, axis=1, keepdims=True) + 1e-9)
sim = F_n @ F_n.T
np.fill_diagonal(sim, -1)

for _ in range(3):
    q = random.randrange(len(paths))
    nn = np.argsort(-sim[q])[:5]
    print(f'\nquery: {paths[q].split("/")[-1]}')
    for i in nn:
        print(f'  sim={sim[q,i]:+.3f}  {paths[i].split("/")[-1]}')

## 7b. Kinetics-fallback embeddings (if SSL collapsed)

The proposal's planned fallback: re-embed everything using R(2+1)D-18 with Kinetics-400 pretrained weights, no fine-tuning. Overwrites `data/embeddings.npz`.

In [ ]:
from pathlib import Path
from src.ssl.embed import embed_dir

embed_dir(
    clips_dir=Path('data/clips'),
    ckpt_path=Path('checkpoints/_force_fallback.pt'),  # nonexistent path => skip ckpt load
    out_npz=Path('data/embeddings.npz'),
    device='cuda',
    batch_size=16,
    kinetics_init=True,
)
print('re-run cell 7 to verify the new embeddings are not collapsed')

## 8. Build all milestone figures

Five PNGs: training curves, geometric-d histogram, two t-SNEs (by fence type + by source video), and a 2×2 detection-overlay grid.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
from src.viz.tsne import plot_tsne
from src.viz.d_distribution import plot as plot_d
from src.viz.training_curve import plot_curves
from src.viz.detection_examples import render_clip, _save_grid
from src.preprocess.detect import Box, HorseDetector

OUT = Path('milestone/figures'); OUT.mkdir(parents=True, exist_ok=True)
(OUT / 'det').mkdir(parents=True, exist_ok=True)

# Clean the CSVs so plot_tsne never sees mixed-type (NaN + str) labels.
auto = pd.read_csv('data/annotations/auto.csv')
auto['type'] = auto['type'].fillna('unannotated').astype(str)
auto.to_csv('data/annotations/auto_clean.csv', index=False)

data = np.load('data/embeddings.npz', allow_pickle=True)
df_v = pd.DataFrame({
    'clip_id': [Path(p).stem for p in data['paths']],
    'video':   [Path(p).stem.rsplit('_', 1)[0] for p in data['paths']],
})
df_v.to_csv('data/annotations/by_video.csv', index=False)

# 8a. Training curves.
plot_curves(Path('checkpoints/train_log.csv'), OUT / 'training_curve.png')

# 8b. Geometric d histogram (annotated clips only).
plot_d(Path('data/annotations/auto.csv'), OUT / 'd_distribution.png')

# 8c. t-SNE by fence type.
plot_tsne(Path('data/embeddings.npz'),
          Path('data/annotations/auto_clean.csv'),
          color_by='type',
          out_path=OUT / 'tsne_type.png',
          title='t-SNE of clip embeddings (by fence type)')

# 8d. t-SNE by source video — visualizes the venue-clustering finding.
plot_tsne(Path('data/embeddings.npz'),
          Path('data/annotations/by_video.csv'),
          color_by='video',
          out_path=OUT / 'tsne_video.png',
          title='t-SNE colored by source video (venue clustering)')

# 8e. Detection grid: 4 sample clips with horse box + fence box + d overlay.
detector = HorseDetector(weights='yolov8n.pt', device='cuda')
df = pd.read_csv('data/annotations/fences.csv').head(4)
rendered = []
for _, row in df.iterrows():
    clip = Path('data/clips') / f"{row['clip_id']}.mp4"
    if not clip.exists():
        continue
    fence_box = Box(float(row['x1']), float(row['y1']),
                    float(row['x2']), float(row['y2']), label='fence')
    pole = int(row['pole_count']) if pd.notna(row.get('pole_count')) else None
    out_jpg = OUT / 'det' / f"{row['clip_id']}.jpg"
    render_clip(clip, fence_box, pole, detector, out_jpg)
    if out_jpg.exists():
        rendered.append(out_jpg)
_save_grid(rendered, OUT / 'det_grid.png', cols=2)

print('\nall figures done:')
!ls -la milestone/figures/*.png

## 9. Preview figures inline

In [ ]:
from IPython.display import Image, display
import glob
for p in sorted(glob.glob('milestone/figures/*.png')):
    print(p)
    display(Image(p))

## 10. Export figures for Overleaf

Zips the 5 milestone figures to Drive so you can download them and drop into your local `milestone_overleaf/figures/` folder before compiling the PDF on Overleaf.

In [ ]:
import shutil, os
os.makedirs('/content/drive/MyDrive/CS131/milestone_export', exist_ok=True)
shutil.make_archive('/content/drive/MyDrive/CS131/milestone_export/figures',
                    'zip', 'milestone/figures')
print('zipped to /content/drive/MyDrive/CS131/milestone_export/figures.zip')
print('Download from Drive, unzip, and copy the PNGs into milestone_overleaf/figures/ on your laptop.')